# Pyspark Hands-on Lab (Beginner → Advanced)

This hands-on lab is designed to simulate real-world data engineering workflows using PySpark. You will start from the fundamentals of DataFrames and gradually move toward advanced transformations, performance optimization, and handling complex data structures.

By the end of this lab, you will be able to:

- Build scalable data pipelines
- Perform business-level data transformations
- Work with structured and semi-structured data
- Apply window functions and optimization techniques

## Level 1: Basic (Foundation)
**Description:**

This section focuses on building a strong foundation in PySpark. You will learn how to create and manipulate DataFrames, which are the core abstraction in Spark. The goal is to understand how data is represented, accessed, and transformed at a basic level.

You will also become familiar with common operations such as filtering, selecting, renaming, and joining datasets—skills that are essential for any data engineer.

**Learning Objectives:** 

- Understand SparkSession and DataFrame basics
- Learn how to read and create datasets
- Perform basic transformations and actions
- Understand schema and data types

**What You Will Practice:**

- Creating DataFrames from files and in-memory data
- Exploring data using schema and descriptive functions
- Filtering rows based on conditions
- Adding, renaming, and dropping columns
- Sorting data for analysis
- Performing basic joins between datasets

## Import Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, concat, concat_ws, lower, \
                                    upper, current_date, sum, avg, min, max, count, when, \
                                    year, month, day, date_sub


from pyspark.sql.types import StructType, StructField, ArrayType, StringType, IntegerType

## Creating Spark Session

In [2]:
spark = SparkSession.builder.appName("MyApp").getOrCreate()

26/03/30 09:12:59 WARN Utils: Your hostname, RAMJANs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.33 instead (on interface en0)
26/03/30 09:12:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 09:13:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

## Create Data Frame

In [4]:
data = [
    (1001, 101, "Alice","Lee", "2026-03-26", 50000,  "F"),
    (1002, 101, "Mac", "Jackwel", "2026-03-26", 60000, "F"),
    (1003, 102, "Max Well","Bonda", "2026-03-26", 75000, "M"),
    (1004, 102, "Robin", "Hunda", "2026-03-26", 56000, "M"),
    (1005, 103, "Ronaldo", "Chap", "2026-02-25", 50000,  "M"),
    (1006, 104, "Messi", "Chap", "2026-02-25", 55000, "M"),
    (1007, 000, "Qqwert", " ", "2026-03-26", -100, "M")
]
cols = ["emp_id", "dept_id", "first_name","last_name", "DOJ", "salary", "gender"]


In [5]:
emp_df = spark.createDataFrame(data=data, schema=cols)
emp_df.show()

+------+-------+----------+---------+----------+------+------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|
+------+-------+----------+---------+----------+------+------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|
|  1007|      0|    Qqwert|         |2026-03-26|  -100|     M|
+------+-------+----------+---------+----------+------+------+



## Explore Data

In [6]:
emp_df.printSchema()

root
 |-- emp_id: long (nullable = true)
 |-- dept_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- DOJ: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- gender: string (nullable = true)



In [7]:
emp_df.describe().show(truncate=False)

26/03/30 09:13:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+-----------------+----------+---------+----------+------------------+------+
|summary|emp_id           |dept_id          |first_name|last_name|DOJ       |salary            |gender|
+-------+-----------------+-----------------+----------+---------+----------+------------------+------+
|count  |7                |7                |7         |7        |7         |7                 |7     |
|mean   |1004.0           |87.57142857142857|NULL      |NULL     |NULL      |49414.28571428572 |NULL  |
|stddev |2.160246899469287|38.63011408584906|NULL      |NULL     |NULL      |23429.356839332188|NULL  |
|min    |1001             |0                |Alice     |         |2026-02-25|-100              |F     |
|max    |1007             |104              |Ronaldo   |Lee      |2026-03-26|75000             |M     |
+-------+-----------------+-----------------+----------+---------+----------+------------------+------+



## Combine two columns into single column & drop 

In [8]:
emp_df.withColumn('full_name', concat(col('first_name'), lit(' '), col('last_name'))) \
        .drop(col('first_name'), col('last_name')) \
        .show()

+------+-------+----------+------+------+--------------+
|emp_id|dept_id|       DOJ|salary|gender|     full_name|
+------+-------+----------+------+------+--------------+
|  1001|    101|2026-03-26| 50000|     F|     Alice Lee|
|  1002|    101|2026-03-26| 60000|     F|   Mac Jackwel|
|  1003|    102|2026-03-26| 75000|     M|Max Well Bonda|
|  1004|    102|2026-03-26| 56000|     M|   Robin Hunda|
|  1005|    103|2026-02-25| 50000|     M|  Ronaldo Chap|
|  1006|    104|2026-02-25| 55000|     M|    Messi Chap|
|  1007|      0|2026-03-26|  -100|     M|      Qqwert  |
+------+-------+----------+------+------+--------------+



In [9]:
emp_df.withColumn('full_name', concat_ws(" ", col('first_name'), col('last_name'))) \
        .drop(col('first_name'), col('last_name')) \
        .show()

+------+-------+----------+------+------+--------------+
|emp_id|dept_id|       DOJ|salary|gender|     full_name|
+------+-------+----------+------+------+--------------+
|  1001|    101|2026-03-26| 50000|     F|     Alice Lee|
|  1002|    101|2026-03-26| 60000|     F|   Mac Jackwel|
|  1003|    102|2026-03-26| 75000|     M|Max Well Bonda|
|  1004|    102|2026-03-26| 56000|     M|   Robin Hunda|
|  1005|    103|2026-02-25| 50000|     M|  Ronaldo Chap|
|  1006|    104|2026-02-25| 55000|     M|    Messi Chap|
|  1007|      0|2026-03-26|  -100|     M|      Qqwert  |
+------+-------+----------+------+------+--------------+



## Add column

In [10]:
emp_df = emp_df.withColumn("Bonus", col('salary') * .3) \
                .withColumn("Country", lit("India"))
emp_df.show()

+------+-------+----------+---------+----------+------+------+-------+-------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|
+------+-------+----------+---------+----------+------+------+-------+-------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|
|  1007|      0|    Qqwert|         |2026-03-26|  -100|     M|  -30.0|  India|
+------+-------+----------+---------+----------+------+------+-------+-------+



## Apply Filter

In [11]:
emp_df.filter(col('DOJ') <= current_date() -1 ) \
        .show()

+------+-------+----------+---------+----------+------+------+-------+-------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|
+------+-------+----------+---------+----------+------+------+-------+-------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|
|  1007|      0|    Qqwert|         |2026-03-26|  -100|     M|  -30.0|  India|
+------+-------+----------+---------+----------+------+------+-------+-------+



In [12]:
emp_df.filter(col('salary') > 0) \
        .groupBy(col('dept_id')) \
        .agg(sum(col('salary')).alias('dept_wise_total_salary')) \
        .orderBy(col('dept_wise_total_salary').desc()) \
        .filter( (col('dept_wise_total_salary') > 50000) & (col('dept_wise_total_salary') < 200000) ) \
        .show()

+-------+----------------------+
|dept_id|dept_wise_total_salary|
+-------+----------------------+
|    102|                131000|
|    101|                110000|
|    104|                 55000|
+-------+----------------------+



## Sorting

In [13]:
emp_df.orderBy(col('salary').asc()).show()

+------+-------+----------+---------+----------+------+------+-------+-------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|
+------+-------+----------+---------+----------+------+------+-------+-------+
|  1007|      0|    Qqwert|         |2026-03-26|  -100|     M|  -30.0|  India|
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|
+------+-------+----------+---------+----------+------+------+-------+-------+



## Creating another Data Frame

In [14]:
data = [
    (101, "Data Engineer"),
    (102, "Software Developer"),
    (103, "Prompt Engineer"),
    (104, "Human Resource"),
    (104, "Human Resource")
]
cols = ["dept_id", "dept_name"]

In [15]:
dept_df = spark.createDataFrame(data, schema=cols)
dept_df.select(col('dept_id'), upper(col('dept_name')).alias('dept_name')).show()

+-------+------------------+
|dept_id|         dept_name|
+-------+------------------+
|    101|     DATA ENGINEER|
|    102|SOFTWARE DEVELOPER|
|    103|   PROMPT ENGINEER|
|    104|    HUMAN RESOURCE|
|    104|    HUMAN RESOURCE|
+-------+------------------+



## All joins

### Inner join
**Note :** Inner join will give all records which have common key in both tables.

In [16]:
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, 'inner') \
        .drop(dept_df.dept_id).show()

+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|         dept_name|
+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|     Data Engineer|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|     Data Engineer|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|Software Developer|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|Software Developer|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|   Prompt Engineer|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|    Human Resource|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|    Human Resource|
+------+-------+----

### Left Join
**Note :** Left join will give the result of all records from left which matched with righ table's key. If those records will not match with right table's key then it'll give records/data from left but right side data will be null.

In [17]:
df = emp_df.join(dept_df, emp_df.dept_id==dept_df.dept_id, 'left') \
        .drop(dept_df.dept_id)

df.show()

+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|         dept_name|
+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|     Data Engineer|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|     Data Engineer|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|Software Developer|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|Software Developer|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|   Prompt Engineer|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|    Human Resource|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|    Human Resource|
|  1007|      0|    

### Handle Null

In [18]:
df.fillna({'dept_name': " "}).show()

+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|         dept_name|
+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|     Data Engineer|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|     Data Engineer|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|Software Developer|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|Software Developer|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|   Prompt Engineer|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|    Human Resource|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|    Human Resource|
|  1007|      0|    

### Drop duplicates

In [19]:
df.dropDuplicates(['emp_id']).show()

+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|         dept_name|
+------+-------+----------+---------+----------+------+------+-------+-------+------------------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|     Data Engineer|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|     Data Engineer|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|Software Developer|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|Software Developer|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|   Prompt Engineer|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|    Human Resource|
|  1007|      0|    Qqwert|         |2026-03-26|  -100|     M|  -30.0|  India|              NULL|
+------+-------+----

### Left Semi
**Note:** The left semi join will give recods from left and columns form left only, for those left keys matches with right keys. It won't give the records, which left keys not matching with right keys with null from right as well it will protect to dupcate records. If you see

In [20]:
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, 'left_semi') \
        .show()

+------+-------+----------+---------+----------+------+------+-------+-------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|  Bonus|Country|
+------+-------+----------+---------+----------+------+------+-------+-------+
|  1001|    101|     Alice|      Lee|2026-03-26| 50000|     F|15000.0|  India|
|  1002|    101|       Mac|  Jackwel|2026-03-26| 60000|     F|18000.0|  India|
|  1003|    102|  Max Well|    Bonda|2026-03-26| 75000|     M|22500.0|  India|
|  1004|    102|     Robin|    Hunda|2026-03-26| 56000|     M|16800.0|  India|
|  1005|    103|   Ronaldo|     Chap|2026-02-25| 50000|     M|15000.0|  India|
|  1006|    104|     Messi|     Chap|2026-02-25| 55000|     M|16500.0|  India|
+------+-------+----------+---------+----------+------+------+-------+-------+



### Left Anti
**Note:** Left anti join will give the records from left which keys not matching right keys only without null from right table.

In [21]:
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, 'leftanti') \
        .show()

+------+-------+----------+---------+----------+------+------+-----+-------+
|emp_id|dept_id|first_name|last_name|       DOJ|salary|gender|Bonus|Country|
+------+-------+----------+---------+----------+------+------+-----+-------+
|  1007|      0|    Qqwert|         |2026-03-26|  -100|     M|-30.0|  India|
+------+-------+----------+---------+----------+------+------+-----+-------+



# Level 2: Intermediate (Business Transformations)
## Description:

In this section, you will move beyond basic transformations and start working on business-level data processing. You will learn how to aggregate data, apply business logic, and generate meaningful insights.

This level introduces real-world scenarios such as calculating customer spending, filtering high-value users, and handling missing or inconsistent data.

**Learning Objectives:**
- Perform aggregations and group-based analysis
- Apply conditional logic to transform data
- Handle missing and duplicate data
- Work with date and time functions

**What You Will Practice:**
- Grouping data and calculating metrics like sum, average, and count
- Filtering aggregated results
- Applying business rules using conditional logic
- Cleaning data by handling null values
- Extracting insights from date fields (year, month, etc.)
- Combining joins with aggregations for deeper analysis

## Created employees Data

In [22]:
data = [
 (1, 'James', 'IT', 80000 ), 
 (2, 'Taylor', 'IT', 80000 ),
 (3, 'Pamela', 'HR', 50000 ),
 (4, 'Sara', 'HR', 40000 ),
 (5, 'David', 'Marketing', 35000 ), 
 (6, 'Smith', 'HR', 65000 ),
 (7, 'Ben', 'HR', 65000 ),
 (8, 'Stokes', 'IT', 45000 ),
 (9, 'David', 'IT', 35000 ), 
 (10, 'Smith', 'HR', 65000 ),
 (11, 'Smith', 'HR', 75000 ), 
 (12, 'Ben', 'Marketing', 65000 ), 
 (13, 'Stokes', 'IT', 35000 ), 
 (14, 'David', 'Marketing', 95000 ), 
 (15, 'Smith', 'HR', 85000 ), 
 (16, 'John', 'Marketing', 68000 )
]
cols = ["emp_id", "emp_name", "dept_name", "salary"]

In [23]:
emp_dataframe = spark.createDataFrame(data=data, schema=cols)
emp_dataframe.show(truncate=False)

+------+--------+---------+------+
|emp_id|emp_name|dept_name|salary|
+------+--------+---------+------+
|1     |James   |IT       |80000 |
|2     |Taylor  |IT       |80000 |
|3     |Pamela  |HR       |50000 |
|4     |Sara    |HR       |40000 |
|5     |David   |Marketing|35000 |
|6     |Smith   |HR       |65000 |
|7     |Ben     |HR       |65000 |
|8     |Stokes  |IT       |45000 |
|9     |David   |IT       |35000 |
|10    |Smith   |HR       |65000 |
|11    |Smith   |HR       |75000 |
|12    |Ben     |Marketing|65000 |
|13    |Stokes  |IT       |35000 |
|14    |David   |Marketing|95000 |
|15    |Smith   |HR       |85000 |
|16    |John    |Marketing|68000 |
+------+--------+---------+------+



## Add column

In [24]:
emp_dataframe = emp_dataframe.withColumn('DOJ', current_date())
emp_dataframe.show()

+------+--------+---------+------+----------+
|emp_id|emp_name|dept_name|salary|       DOJ|
+------+--------+---------+------+----------+
|     1|   James|       IT| 80000|2026-03-30|
|     2|  Taylor|       IT| 80000|2026-03-30|
|     3|  Pamela|       HR| 50000|2026-03-30|
|     4|    Sara|       HR| 40000|2026-03-30|
|     5|   David|Marketing| 35000|2026-03-30|
|     6|   Smith|       HR| 65000|2026-03-30|
|     7|     Ben|       HR| 65000|2026-03-30|
|     8|  Stokes|       IT| 45000|2026-03-30|
|     9|   David|       IT| 35000|2026-03-30|
|    10|   Smith|       HR| 65000|2026-03-30|
|    11|   Smith|       HR| 75000|2026-03-30|
|    12|     Ben|Marketing| 65000|2026-03-30|
|    13|  Stokes|       IT| 35000|2026-03-30|
|    14|   David|Marketing| 95000|2026-03-30|
|    15|   Smith|       HR| 85000|2026-03-30|
|    16|    John|Marketing| 68000|2026-03-30|
+------+--------+---------+------+----------+



## Filter last one week data

In [25]:
emp_dataframe = emp_dataframe.withColumn('year', year('DOJ') ) \
                .withColumn('month', month('DOJ')) \
                .withColumn('day', day('DOJ'))

emp_dataframe.show()

+------+--------+---------+------+----------+----+-----+---+
|emp_id|emp_name|dept_name|salary|       DOJ|year|month|day|
+------+--------+---------+------+----------+----+-----+---+
|     1|   James|       IT| 80000|2026-03-30|2026|    3| 30|
|     2|  Taylor|       IT| 80000|2026-03-30|2026|    3| 30|
|     3|  Pamela|       HR| 50000|2026-03-30|2026|    3| 30|
|     4|    Sara|       HR| 40000|2026-03-30|2026|    3| 30|
|     5|   David|Marketing| 35000|2026-03-30|2026|    3| 30|
|     6|   Smith|       HR| 65000|2026-03-30|2026|    3| 30|
|     7|     Ben|       HR| 65000|2026-03-30|2026|    3| 30|
|     8|  Stokes|       IT| 45000|2026-03-30|2026|    3| 30|
|     9|   David|       IT| 35000|2026-03-30|2026|    3| 30|
|    10|   Smith|       HR| 65000|2026-03-30|2026|    3| 30|
|    11|   Smith|       HR| 75000|2026-03-30|2026|    3| 30|
|    12|     Ben|Marketing| 65000|2026-03-30|2026|    3| 30|
|    13|  Stokes|       IT| 35000|2026-03-30|2026|    3| 30|
|    14|   David|Marketi

In [26]:
emp_dataframe.filter(col('DOJ') >= date_sub(current_date(), 7)).show()

+------+--------+---------+------+----------+----+-----+---+
|emp_id|emp_name|dept_name|salary|       DOJ|year|month|day|
+------+--------+---------+------+----------+----+-----+---+
|     1|   James|       IT| 80000|2026-03-30|2026|    3| 30|
|     2|  Taylor|       IT| 80000|2026-03-30|2026|    3| 30|
|     3|  Pamela|       HR| 50000|2026-03-30|2026|    3| 30|
|     4|    Sara|       HR| 40000|2026-03-30|2026|    3| 30|
|     5|   David|Marketing| 35000|2026-03-30|2026|    3| 30|
|     6|   Smith|       HR| 65000|2026-03-30|2026|    3| 30|
|     7|     Ben|       HR| 65000|2026-03-30|2026|    3| 30|
|     8|  Stokes|       IT| 45000|2026-03-30|2026|    3| 30|
|     9|   David|       IT| 35000|2026-03-30|2026|    3| 30|
|    10|   Smith|       HR| 65000|2026-03-30|2026|    3| 30|
|    11|   Smith|       HR| 75000|2026-03-30|2026|    3| 30|
|    12|     Ben|Marketing| 65000|2026-03-30|2026|    3| 30|
|    13|  Stokes|       IT| 35000|2026-03-30|2026|    3| 30|
|    14|   David|Marketi

## Apply Aggregate functions

In [27]:
df = emp_dataframe.groupBy(col('dept_name')) \
    .agg(
        count(col('emp_id')).alias('dept_wise_emp_count'),
        min(col('salary')).alias('dept_wise_min_salary'),
        max(col('salary')).alias('dept_wise_max_salary'),
        avg(col('salary')).alias('dept_wise_avg_salary'),
        sum(col('salary')).alias('dept_wise_total_salary')
    ) \
    .orderBy(col('dept_wise_total_salary').desc()) 

df.show()

+---------+-------------------+--------------------+--------------------+--------------------+----------------------+
|dept_name|dept_wise_emp_count|dept_wise_min_salary|dept_wise_max_salary|dept_wise_avg_salary|dept_wise_total_salary|
+---------+-------------------+--------------------+--------------------+--------------------+----------------------+
|       HR|                  7|               40000|               85000|   63571.42857142857|                445000|
|       IT|                  5|               35000|               80000|             55000.0|                275000|
|Marketing|                  4|               35000|               95000|             65750.0|                263000|
+---------+-------------------+--------------------+--------------------+--------------------+----------------------+



## Use of case / when

In [28]:
df.withColumn("paid_category",
             when(col('dept_wise_avg_salary') > 64000, "High Paid")
              .when(col('dept_wise_avg_salary') > 55000, "Medium Paid")
              .otherwise("Low Paid")
             ).orderBy(col('dept_wise_avg_salary').desc()) \
            .show()

+---------+-------------------+--------------------+--------------------+--------------------+----------------------+-------------+
|dept_name|dept_wise_emp_count|dept_wise_min_salary|dept_wise_max_salary|dept_wise_avg_salary|dept_wise_total_salary|paid_category|
+---------+-------------------+--------------------+--------------------+--------------------+----------------------+-------------+
|Marketing|                  4|               35000|               95000|             65750.0|                263000|    High Paid|
|       HR|                  7|               40000|               85000|   63571.42857142857|                445000|  Medium Paid|
|       IT|                  5|               35000|               80000|             55000.0|                275000|     Low Paid|
+---------+-------------------+--------------------+--------------------+--------------------+----------------------+-------------+



# Level 3: Advanced (Industry-Level Processing)
## Description:

This section focuses on advanced PySpark capabilities that are widely used in real-world data engineering projects. You will learn how to work with large-scale datasets, apply window functions, and handle complex data types such as nested JSON and arrays.

You will also explore performance optimization techniques to ensure your Spark jobs run efficiently in production environments.

**Learning Objectives:**

- Master window functions for advanced analytics
- Work with semi-structured and nested data
- Optimize Spark jobs for performance
- Understand distributed data processing concepts

**What You Will Practice:**
- Ranking and partitioning data using window functions
- Calculating running totals and trends
- Using lag and lead for time-based analysis
- Reading and writing different file formats (CSV, JSON, Parquet)
- Handling nested structures and arrays
- Optimizing performance using caching and partitioning
- Using broadcast joins for efficient data processing

## Window or Analytics functions

In [29]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number, sum , lag, lead

In [30]:
windowSpec = Window.partitionBy(col('dept_name')).orderBy(col('salary').desc())

In [31]:
emp_dataframe.withColumn("rank", rank().over(windowSpec)) \
            .withColumn('dense_rank', dense_rank().over(windowSpec)) \
            .withColumn('row_number', row_number().over(windowSpec)) \
            .withColumn('prev_salary', lag(col("salary")).over(windowSpec)) \
            .withColumn('next_salary', lead(col("salary")).over(windowSpec)) \
            .withColumn('running_sum', sum(col('salary')).over(windowSpec.rowsBetween(Window.unboundedPreceding, Window.currentRow))) \
            .show()

+------+--------+---------+------+----------+----+-----+---+----+----------+----------+-----------+-----------+-----------+
|emp_id|emp_name|dept_name|salary|       DOJ|year|month|day|rank|dense_rank|row_number|prev_salary|next_salary|running_sum|
+------+--------+---------+------+----------+----+-----+---+----+----------+----------+-----------+-----------+-----------+
|    15|   Smith|       HR| 85000|2026-03-30|2026|    3| 30|   1|         1|         1|       NULL|      75000|      85000|
|    11|   Smith|       HR| 75000|2026-03-30|2026|    3| 30|   2|         2|         2|      85000|      65000|     160000|
|     6|   Smith|       HR| 65000|2026-03-30|2026|    3| 30|   3|         3|         3|      75000|      65000|     225000|
|     7|     Ben|       HR| 65000|2026-03-30|2026|    3| 30|   3|         3|         4|      65000|      65000|     290000|
|    10|   Smith|       HR| 65000|2026-03-30|2026|    3| 30|   3|         3|         5|      65000|      50000|     355000|
|     3|

## Array type data

In [32]:
from pyspark.sql.functions import explode

In [33]:
schema = StructType([
    StructField("dept_id", IntegerType(), True),
    StructField("e_id", ArrayType(IntegerType()), True)
])

In [34]:
data = [
    {"dept_id":101, "e_id":[10101, 10102, 10103]},
    {"dept_id": 102, "e_id":[10201, 10202]}
]


In [35]:
struct_df = spark.createDataFrame(data, schema=schema)
struct_df.show(truncate=False)

+-------+---------------------+
|dept_id|e_id                 |
+-------+---------------------+
|101    |[10101, 10102, 10103]|
|102    |[10201, 10202]       |
+-------+---------------------+



In [36]:
df_exploded = struct_df.withColumn("e_id", explode(struct_df["e_id"]))

In [37]:
df_exploded.show()

+-------+-----+
|dept_id| e_id|
+-------+-----+
|    101|10101|
|    101|10102|
|    101|10103|
|    102|10201|
|    102|10202|
+-------+-----+



## Nested Data

In [38]:
data = [
    (1, {"name": "Alice", "age": 25}),
    (2, {"name": "John", "age": 30}),
    (3, {"name": "Denial", "age": 32, "City": "New York"})
]
df = spark.createDataFrame(data, ["id", "user"])

df.show(truncate=False)

+---+---------------------------------------------+
|id |user                                         |
+---+---------------------------------------------+
|1  |{name -> Alice, age -> 25}                   |
|2  |{name -> John, age -> 30}                    |
|3  |{name -> Denial, City -> New York, age -> 32}|
+---+---------------------------------------------+



In [39]:
df.select("id", "user.name", "user.age", "user.City").show()

+---+------+---+--------+
| id|  name|age|    City|
+---+------+---+--------+
|  1| Alice| 25|    NULL|
|  2|  John| 30|    NULL|
|  3|Denial| 32|New York|
+---+------+---+--------+



In [40]:
df.select(col('id'), col('user.name'), col('user.age'), col('user.City')).show()

+---+------+---+--------+
| id|  name|age|    City|
+---+------+---+--------+
|  1| Alice| 25|    NULL|
|  2|  John| 30|    NULL|
|  3|Denial| 32|New York|
+---+------+---+--------+



## Complex Nested

In [41]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, FloatType, ArrayType

In [42]:
schema = StructType([
    StructField("order_no", StringType()),
    StructField("order_date", StringType()),
    StructField("order_lines", ArrayType(
        StructType([
                    StructField("item_id", StringType()),
                    StructField("item_unit_price", FloatType()),
                    StructField("item_qty", IntegerType()),
                    StructField("item_selling_price", FloatType()),
                    StructField("discounts", ArrayType(
                        StructType([
                            StructField("discount_amt", FloatType()),
                            StructField("discount_name", StringType())
                        ])
                    ))
        ])
    ))
])

In [43]:
data = [
    {
    "order_no":"ORD123",
    "order_date":"2023-10-01",
    "order_lines": [
        {
            "item_id":"ITM001",
            "item_unit_price":100.0,
            "item_qty":2,
            "item_selling_price":180.0,
            "discounts":[
                {
                    "discount_amt":10.0,
                    "discount_name":"Diwali_Offer"
                },
                {
                    "discount_amt":10.0,
                    "discount_name":"Loyalty"
                }
            ]
        },
        {
            "item_id":"ITEM002",
            "item_unit_price":50.0,
            "item_qty": 1,
            "item_selling_price":45.0,
            "discounts":[]
        }
    ]
}
]

In [44]:
df = spark.createDataFrame(data=data, schema=schema)
df.show(truncate=False)

+--------+----------+--------------------------------------------------------------------------------------------------+
|order_no|order_date|order_lines                                                                                       |
+--------+----------+--------------------------------------------------------------------------------------------------+
|ORD123  |2023-10-01|[{ITM001, 100.0, 2, 180.0, [{10.0, Diwali_Offer}, {10.0, Loyalty}]}, {ITEM002, 50.0, 1, 45.0, []}]|
+--------+----------+--------------------------------------------------------------------------------------------------+



In [45]:
df = df.withColumn("order_line", explode(col("order_lines")))

df.select(col("order_no"), col("order_date"), col("order_line.item_id"), 
              col("order_line.item_unit_price"), col("order_line.item_qty"),
              col("order_line.discounts")).show(truncate=False)


+--------+----------+-------+---------------+--------+---------------------------------------+
|order_no|order_date|item_id|item_unit_price|item_qty|discounts                              |
+--------+----------+-------+---------------+--------+---------------------------------------+
|ORD123  |2023-10-01|ITM001 |100.0          |2       |[{10.0, Diwali_Offer}, {10.0, Loyalty}]|
|ORD123  |2023-10-01|ITEM002|50.0           |1       |[]                                     |
+--------+----------+-------+---------------+--------+---------------------------------------+



In [46]:
df = df.withColumn("discount", explode(col("order_line.discounts")))
df.select(col("order_no"), col("order_date"), col("order_line.item_id"), 
              col("order_line.item_unit_price"), col("order_line.item_qty"),
              col("discount.discount_amt"), col("discount.discount_name")).show(truncate=False)

+--------+----------+-------+---------------+--------+------------+-------------+
|order_no|order_date|item_id|item_unit_price|item_qty|discount_amt|discount_name|
+--------+----------+-------+---------------+--------+------------+-------------+
|ORD123  |2023-10-01|ITM001 |100.0          |2       |10.0        |Diwali_Offer |
|ORD123  |2023-10-01|ITM001 |100.0          |2       |10.0        |Loyalty      |
+--------+----------+-------+---------------+--------+------------+-------------+



In [47]:
df.select("order_no", "order_date", "order_line.item_id", 
          "order_line.item_unit_price", "order_line.item_qty",  "discount.discount_amt", "discount.discount_name") \
            .show(truncate=False)

+--------+----------+-------+---------------+--------+------------+-------------+
|order_no|order_date|item_id|item_unit_price|item_qty|discount_amt|discount_name|
+--------+----------+-------+---------------+--------+------------+-------------+
|ORD123  |2023-10-01|ITM001 |100.0          |2       |10.0        |Diwali_Offer |
|ORD123  |2023-10-01|ITM001 |100.0          |2       |10.0        |Loyalty      |
+--------+----------+-------+---------------+--------+------------+-------------+

